In [ ]:
import torch

# 1. Tải các trọng số của mô hình
state_dict = torch.load(r'D:\nids-vae-project\artifacts\models\vae_best.pth')

# 2. In ra các khóa (keys) trong dictionary để kiểm tra
print(state_dict.keys())

In [ ]:
for name, param in state_dict.items():
    print(f"{name}: {param.shape}")

In [ ]:
print(state_dict['encoder.0.weight'])

# ý nghĩa 128 hàng và 66 cột

In [ ]:
w_h1 = state_dict['encoder.0.weight'][0]

print(w_h1)

# đây chính là vector trọng số của 1 neuron trong layer đầu tiên của encoder, có 66 chiều
#  (tương ứng với 66 input features). Mỗi giá trị trong vector này đại diện cho trọng số kết nối 
# từ một input feature đến neuron đó.

In [ ]:
print(
    state_dict['encoder.0.bias'][0]
)
# đây là trọng số bias của neuron đầu tiên trong layer đầu tiên của encoder, có giá trị là -0.0028. 
# Trọng số bias này sẽ được cộng vào tổng trọng số kết nối từ các input features trước khi áp dụng hàm
#  kích hoạt (activation function) trong quá trình tính toán output của neuron đó.

In [ ]:
print(
    state_dict['fc_mu.weight']
)
# đây là trọng số của layer fully connected (fc) đầu tiên trong phần encoder, có shape (16, 128).
# Điều này có nghĩa là layer này có 16 output features (tương ứng với 16 chiều của latent space) và
#  nhận đầu vào từ 128 neurons của layer trước đó trong encoder.
#  Mỗi giá trị trong ma trận trọng số này đại diện cho trọng số kết nối giữa một neuron ở layer 
# trước đó và một neuron ở layer fully connected này.


In [ ]:
# ============================================================

# TẠO FILE DEMO ĐẦY ĐỦ CÁC KIỂU TẤN CÔNG TỪ CICIDS2017 RAW

# Output: data/demo/demo_all_attack_types.csv

# ============================================================

import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------

# 1. Cấu hình

# ------------------------------------------------------------

SAMPLES_PER_CLASS = 10
RANDOM_SEED = 42

TARGET_LABELS = [
"BENIGN",
"DDoS",
"PortScan",
"Bot",
"Infiltration",
"DoS Hulk",
"DoS GoldenEye",
"DoS slowloris",
"DoS Slowhttptest",
"Heartbleed",
"FTP-Patator",
"SSH-Patator",
"Web Attack - Brute Force",
"Web Attack - XSS",
"Web Attack - Sql Injection",
]

rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------

# 2. Tìm project root

# ------------------------------------------------------------

def find_project_root(start: Path) -> Path:
"""Tìm thư mục project dựa trên sự tồn tại của data/ và artifacts/."""
for candidate in [start.resolve(), *start.resolve().parents]:
if (candidate / "data").exists() and (candidate / "artifacts").exists():
return candidate

```
raise FileNotFoundError(
    "Không tìm thấy project root. "
    "Hãy chạy notebook từ bên trong thư mục dự án NIDS-VAE."
)
```

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

OUTPUT_DIR = DATA_DIR / "demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "demo_all_attack_types.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"File output : {OUTPUT_CSV}")

# ------------------------------------------------------------

# 3. Đọc feature schema 66 cột đúng với mô hình

# ------------------------------------------------------------

def load_feature_columns(project_root: Path) -> list[str]:
"""
Tìm feature_columns.json trong project.
File này đảm bảo CSV demo có đúng thứ tự 66 feature mà backend yêu cầu.
"""
candidates = list(project_root.rglob("feature_columns.json"))

```
if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy feature_columns.json. "
        "Hãy chạy clean_data.py trước để sinh artifact feature schema."
    )

# Ưu tiên file nằm trong artifacts/
candidates.sort(
    key=lambda p: (
        0 if "artifacts" in p.parts else 1,
        len(str(p)),
    )
)

feature_path = candidates[0]

with feature_path.open("r", encoding="utf-8") as f:
    payload = json.load(f)

if isinstance(payload, list):
    feature_columns = payload

elif isinstance(payload, dict):
    feature_columns = (
        payload.get("feature_columns")
        or payload.get("features")
        or payload.get("columns")
    )

else:
    feature_columns = None

if not isinstance(feature_columns, list) or not feature_columns:
    raise ValueError(
        f"Không đọc được danh sách feature từ: {feature_path}"
    )

feature_columns = [str(col).strip() for col in feature_columns]

print(f"Feature schema: {feature_path}")
print(f"Số feature dùng cho demo: {len(feature_columns)}")

return feature_columns
```

FEATURE_COLUMNS = load_feature_columns(PROJECT_ROOT)

if len(FEATURE_COLUMNS) != 66:
print(
f"Cảnh báo: feature schema hiện có {len(FEATURE_COLUMNS)} cột, "
"không phải 66. Code vẫn tiếp tục dùng schema hiện tại."
)

# ------------------------------------------------------------

# 4. Tìm các file raw CICIDS2017

# ------------------------------------------------------------

def find_raw_csv_files(project_root: Path, output_csv: Path) -> list[Path]:
"""
Ưu tiên các file nằm trong data/raw, data/raw_data, data/cicids2017.
Nếu không có, tìm file có tên chứa pcap hoặc ISCX.
"""
preferred_dirs = [
project_root / "data" / "raw",
project_root / "data" / "raw_data",
project_root / "data" / "cicids2017",
project_root / "data" / "CICIDS2017",
]

```
raw_files = []

for raw_dir in preferred_dirs:
    if raw_dir.exists():
        raw_files.extend(raw_dir.rglob("*.csv"))

if not raw_files:
    for path in project_root.rglob("*.csv"):
        name = path.name.lower()

        if (
            "pcap" in name
            or "iscx" in name
            or "workinghours" in name
        ):
            raw_files.append(path)

# Loại file demo, test đã xử lý, artifact và file output hiện tại
excluded_keywords = [
    "demo",
    "sample",
    "metadata",
    "processed",
    "train",
    "val",
    "test",
]

clean_files = []

for path in raw_files:
    path_resolved = path.resolve()
    path_text = str(path_resolved).lower()

    if path_resolved == output_csv.resolve():
        continue

    if "artifacts" in path_text:
        continue

    if any(keyword in path.name.lower() for keyword in excluded_keywords):
        continue

    clean_files.append(path_resolved)

return sorted(set(clean_files))
```

RAW_CSV_FILES = find_raw_csv_files(PROJECT_ROOT, OUTPUT_CSV)

if not RAW_CSV_FILES:
raise FileNotFoundError(
"Không tìm thấy file raw CICIDS2017. "
"Kiểm tra lại các thư mục data/raw, data/raw_data hoặc data/cicids2017."
)

print("\nCác file raw được sử dụng:")
for path in RAW_CSV_FILES:
print(f" - {path.relative_to(PROJECT_ROOT)}")

# ------------------------------------------------------------

# 5. Chuẩn hóa label CICIDS2017 về 14 kiểu attack chuẩn

# ------------------------------------------------------------

def normalize_text(value: str) -> str:
"""Chuẩn hóa text để xử lý khoảng trắng, dấu gạch và ký tự lỗi encoding."""
text = str(value).strip()

```
text = unicodedata.normalize("NFKD", text)
text = text.replace("\ufeff", "")
text = text.replace("�", " ")
text = re.sub(r"[–—−]", "-", text)
text = re.sub(r"\s+", " ", text)

return text.strip().lower()
```

def canonical_attack_label(raw_label: str) -> str | None:
"""
Chuyển label raw CICIDS2017 sang tên chuẩn.
Trả về None nếu label không thuộc nhóm cần lấy.
"""
label = normalize_text(raw_label)

```
if label == "benign":
    return "BENIGN"

if "heartbleed" in label:
    return "Heartbleed"

if "hulk" in label:
    return "DoS Hulk"

if "slowhttptest" in label:
    return "DoS Slowhttptest"

if "slowloris" in label:
    return "DoS slowloris"

if "goldeneye" in label:
    return "DoS GoldenEye"

if "ddos" in label:
    return "DDoS"

if "infiltration" in label:
    return "Infiltration"

if label == "bot" or label.startswith("bot "):
    return "Bot"

if "portscan" in label:
    return "PortScan"

if "ftp-patator" in label or "ftp patator" in label:
    return "FTP-Patator"

if "ssh-patator" in label or "ssh patator" in label:
    return "SSH-Patator"

if "web attack" in label and "brute" in label:
    return "Web Attack - Brute Force"

if "web attack" in label and "xss" in label:
    return "Web Attack - XSS"

if (
    "web attack" in label
    and (
        "sql" in label
        or "injection" in label
    )
):
    return "Web Attack - Sql Injection"

return None
```

# ------------------------------------------------------------

# 6. Đọc raw theo chunks và lấy mẫu thực tế từng attack

# ------------------------------------------------------------

samples_by_label = {label: [] for label in TARGET_LABELS}
files_processed = 0
files_skipped = []

for csv_path in RAW_CSV_FILES:
print(f"\nĐang đọc: {csv_path.name}")

```
try:
    chunk_iterator = pd.read_csv(
        csv_path,
        chunksize=100_000,
        low_memory=False,
        encoding="utf-8",
        encoding_errors="replace",
    )

    for chunk in chunk_iterator:
        # Xóa khoảng trắng đầu/cuối tên cột như clean_data.py
        chunk.columns = [str(col).strip() for col in chunk.columns]

        # Tìm cột nhãn
        label_candidates = [
            col for col in chunk.columns
            if str(col).strip().lower() == "label"
        ]

        if not label_candidates:
            raise KeyError("Không tìm thấy cột Label trong file raw.")

        label_col_raw = label_candidates[0]

        # Bỏ qua file không chứa đủ các feature mà model yêu cầu
        missing_features = [
            feature
            for feature in FEATURE_COLUMNS
            if feature not in chunk.columns
        ]

        if missing_features:
            raise KeyError(
                f"Thiếu {len(missing_features)} feature cần thiết. "
                f"Ví dụ: {missing_features[:5]}"
            )

        # Chỉ giữ đúng feature schema + nhãn
        subset = chunk[FEATURE_COLUMNS + [label_col_raw]].copy()

        # Map raw label về nhãn chuẩn
        subset["_canonical_label"] = subset[label_col_raw].map(
            canonical_attack_label
        )

        # Lấy các loại label đang cần
        subset = subset[
            subset["_canonical_label"].isin(TARGET_LABELS)
        ].copy()

        if subset.empty:
            continue

        # Lấy tối đa SAMPLES_PER_CLASS dòng mỗi label mỗi chunk
        # Sau đó sẽ random lại ở bước cuối.
        for attack_label in TARGET_LABELS:
            rows = subset[
                subset["_canonical_label"] == attack_label
            ]

            if rows.empty:
                continue

            take_n = min(SAMPLES_PER_CLASS, len(rows))

            sampled_rows = rows.sample(
                n=take_n,
                random_state=int(rng.integers(0, 2**31 - 1)),
            )

            samples_by_label[attack_label].append(sampled_rows)

    files_processed += 1

except Exception as error:
    files_skipped.append((csv_path.name, str(error)))
    print(f"  Bỏ qua file vì lỗi: {error}")
```

# ------------------------------------------------------------

# 7. Ghép và lấy đúng số mẫu cho từng label

# ------------------------------------------------------------

final_parts = []
summary_rows = []

for attack_label in TARGET_LABELS:
collected_parts = samples_by_label[attack_label]

```
if not collected_parts:
    summary_rows.append(
        {
            "Loại": attack_label,
            "Số mẫu yêu cầu": SAMPLES_PER_CLASS,
            "Số mẫu thu được": 0,
            "Trạng thái": "Không tìm thấy trong raw data",
        }
    )
    continue

collected_df = pd.concat(collected_parts, ignore_index=True)

# Lấy random đúng số lượng mong muốn; nếu không đủ thì lấy toàn bộ.
take_n = min(SAMPLES_PER_CLASS, len(collected_df))

final_sample = collected_df.sample(
    n=take_n,
    random_state=int(rng.integers(0, 2**31 - 1)),
).copy()

# Giữ tên Label chuẩn, để dễ kiểm tra demo.
final_sample["Label"] = attack_label

# Chỉ giữ feature + Label, không đưa cột phụ vào Dashboard.
final_sample = final_sample[FEATURE_COLUMNS + ["Label"]]

final_parts.append(final_sample)

status = (
    "Đủ số mẫu"
    if take_n == SAMPLES_PER_CLASS
    else "Không đủ số mẫu trong raw data"
)

summary_rows.append(
    {
        "Loại": attack_label,
        "Số mẫu yêu cầu": SAMPLES_PER_CLASS,
        "Số mẫu thu được": take_n,
        "Trạng thái": status,
    }
)
```

if not final_parts:
raise RuntimeError(
"Không tạo được bất kỳ mẫu nào. "
"Hãy kiểm tra raw path, Label và feature_columns.json."
)

demo_all_attacks = pd.concat(final_parts, ignore_index=True)

# Trộn các loại attack để file demo giống dữ liệu thực tế hơn

demo_all_attacks = demo_all_attacks.sample(
frac=1,
random_state=RANDOM_SEED,
).reset_index(drop=True)

# Chuyển Inf thành NaN để backend xử lý bằng median imputation

demo_all_attacks = demo_all_attacks.replace([np.inf, -np.inf], np.nan)

# Lưu file CSV

demo_all_attacks.to_csv(
OUTPUT_CSV,
index=False,
encoding="utf-8-sig",
)

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("HOÀN THÀNH TẠO FILE DEMO")
print("=" * 70)
print(f"Số file raw đã đọc thành công: {files_processed}")
print(f"Tổng số dòng demo          : {len(demo_all_attacks)}")
print(f"Số cột feature             : {len(FEATURE_COLUMNS)}")
print(f"Đường dẫn output           : {OUTPUT_CSV}")

print("\nPhân bố label trong file demo:")
display(
demo_all_attacks["Label"]
.value_counts()
.reindex(TARGET_LABELS, fill_value=0)
.rename_axis("Label")
.reset_index(name="Số flow")
)

print("\nBáo cáo lấy mẫu:")
display(summary_df)

print("\n5 dòng đầu của file demo:")
display(demo_all_attacks.head())

if files_skipped:
print("\nCác file bị bỏ qua:")
for file_name, reason in files_skipped:
print(f" - {file_name}: {reason}")

missing_labels = summary_df.loc[
summary_df["Số mẫu thu được"] == 0,
"Loại",
].tolist()

if missing_labels:
print("\nCẢNH BÁO: Chưa lấy được mẫu cho các loại:")
print(", ".join(missing_labels))
else:
print("\nĐã lấy được đầy đủ BENIGN và 14 loại tấn công.")
